In [101]:
import numpy as np
from itertools import combinations

lstms_stack = np.load('lstms_stack.npy')

n_repeats = 32
n_maps = 32
n_samples = 4501
n_players = 4
n_pairs = n_players * (n_players - 1) // 2
pairs = combinations(np.arange(n_players), 2)
n_pcs = 142
pc_pairs = n_pcs*(n_pcs-1)//2 + n_pcs

In [102]:
lstms_stack = np.moveaxis(lstms_stack, 1, -1)
lstms_stack.shape

(4609024, 142, 4)

In [108]:
from brainiak.isc import isfc
#input shape 4501, 142, 4

isfcs = []
#isfcs = np.full((n_maps*n_repeats, n_pairs, pc_pairs), np.nan, dtype='uint8')
for i in range(n_maps*n_repeats):
    inp = lstms_stack[4501*i:4501*(i+1)]
    fc, isc = isfc(inp, pairwise=True, vectorize_isfcs=True)
    fc = np.concatenate((fc,isc), axis=1)
    isfcs.append(fc)
    if i%10==0:
        print('Step',i,'done')
isfcs = np.array(isfcs)
print(isfcs[0].shape)
isfcs[0]

Step 0 done
Step 10 done
Step 20 done
Step 30 done
Step 40 done
Step 50 done
Step 60 done
Step 70 done
Step 80 done
Step 90 done
Step 100 done
Step 110 done
Step 120 done
Step 130 done
Step 140 done
Step 150 done
Step 160 done
Step 170 done
Step 180 done
Step 190 done
Step 200 done
Step 210 done
Step 220 done
Step 230 done
Step 240 done
Step 250 done
Step 260 done
Step 270 done
Step 280 done
Step 290 done
Step 300 done
Step 310 done
Step 320 done
Step 330 done
Step 340 done
Step 350 done
Step 360 done
Step 370 done
Step 380 done
Step 390 done
Step 400 done
Step 410 done
Step 420 done
Step 430 done
Step 440 done
Step 450 done
Step 460 done
Step 470 done
Step 480 done
Step 490 done
Step 500 done
Step 510 done
Step 520 done
Step 530 done
Step 540 done
Step 550 done
Step 560 done
Step 570 done
Step 580 done
Step 590 done
Step 600 done
Step 610 done
Step 620 done
Step 630 done
Step 640 done
Step 650 done
Step 660 done
Step 670 done
Step 680 done
Step 690 done
Step 700 done
Step 710 done
Ste

array([[ 0.02864637, -0.02826394, -0.02146128, ...,  0.0492506 ,
         0.12292513,  0.00842804],
       [-0.01024522, -0.08894925, -0.08876143, ...,  0.00689716,
         0.09848091, -0.00802883],
       [-0.02761853, -0.06053264, -0.07597508, ...,  0.02077428,
         0.04399599, -0.00324973],
       [-0.04230381, -0.07649878, -0.11273504, ...,  0.04830024,
         0.00566999,  0.01903719],
       [-0.01229955, -0.07648806, -0.10166255, ..., -0.01330514,
         0.02397508, -0.05290515],
       [-0.02669812, -0.04662139, -0.07812543, ...,  0.00015415,
         0.08831362,  0.03483348]])

In [169]:
fc.shape

(6, 10153)

In [112]:
isfcs.shape

(1024, 6, 10153)

In [89]:
isc.shape

(6, 142)

In [91]:
np.concatenate((fc,isc), axis=1).shape

(6, 10153)

In [110]:
np.save('isfcs.npy', isfcs)

In [111]:
isfcs = np.load('isfcs.npy')

In [77]:
np.isnan(isfcs).any()

False

In [168]:
isfcs_stack.shape

(2048, 10153)

In [12]:
map_ids = []
for map_id in np.arange(n_maps):
    map_ids.extend([map_id]*n_repeats)
len(map_ids)

1024

In [170]:
import pandas as pd

isfcs_dict = {'PC Pair': [], 'cooperative isfc': [], 'competitive isfc': []}

coop_ids = (0, 5)
comp_ids = (1, 2, 3, 4)
for pc_pair_id in np.arange(pc_pairs): 
    for map_id in np.arange(n_maps):
        for repeat_id in np.arange(n_repeats):
            game_id=map_id*n_repeats+repeat_id
            coop = np.mean(isfcs[game_id, coop_ids, pc_pair_id], axis=0)
            comp = np.mean(isfcs[game_id, comp_ids, pc_pair_id], axis=0)
            #if not np.isnan(coefs[pc_id, 5, game_id]):
            #print(coefs[pc_id, coop_ids, game_id])
            #coop_coef = np.mean(coefs[pc_id, coop_ids, map_id], axis=0)
            #comp_coef = np.mean(coefs[pc_id, comp_ids, map_id], axis=0)
            isfcs_dict['cooperative isfc'].append(coop)
            isfcs_dict['competitive isfc'].append(comp)
            isfcs_dict['PC Pair'].append(pc_pair_id)

isfcs_df = pd.DataFrame(isfcs_dict)
isfcs_df

KeyboardInterrupt: 

Wins and Scores

In [113]:
wins = np.load(f'/jukebox/hasson/snastase/social-ctf/results/wins_matchup-0.npy')
scores = np.load(f'/jukebox/hasson/snastase/social-ctf/results/scores_matchup-0.npy')

win_stack = np.hstack(np.split(wins, n_repeats, axis=0)).squeeze() #32 x 144032 x 2
print(f"Wins stack shape: {win_stack.shape}")
score_stack = np.hstack(np.split(scores, n_repeats, axis=0)).squeeze() #32 x 144032 x 2
print(f"Scores stack shape: {score_stack.shape}")

Wins stack shape: (1024, 2)
Scores stack shape: (1024, 2)


In [126]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import PredefinedSplit
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score
from scipy.stats import zscore
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

#use set of alphas = [0, 1, 10, 100, 1000, 10000]
alphas = [100, 1000, 10000] #0.1, 1, 10 removed for speed on reruns - initial tests had 1000 consistently best

outerCV = KFold(n_splits=32) 
innerCV = KFold(n_splits=31) 
#cv = PredefinedSplit(map_ids)

scaler = StandardScaler()

print(type(outerCV))

teams = (0,5)
isfcs_stack = np.vstack((isfcs[:, teams[0], :], isfcs[:, teams[1], :]))#.squeeze()[:, np.newaxis]
#print(win_stack[:, 0].shape)
scores_stack = np.concatenate((score_stack[:, 0], score_stack[:, 1]), axis=0)
print(scores_stack.shape)
#raise

results_scores = np.full((n_maps), np.nan)   #n_repeats
coefs_scores = np.full((n_maps, pc_pairs), np.nan)
best_alphas_scores = np.full((n_maps), np.nan)
pairs = [(0, (0,1)), (5, (2,3))]
#for pc_pair in np.arange(pc_pairs):
#for pair_id, pair in pairs:
#scores = []
for f, (train, test) in tqdm(enumerate(outerCV.split(isfcs_stack))):
    clf = RidgeCV(alphas, cv=innerCV) 
    
    isfc_train = scaler.fit_transform(isfcs_stack[train]) 
    isfc_test = scaler.transform(isfcs_stack[test])
    
    #print('fit next')

    '''print(isfcs_stack.shape)
    print(np.isnan(isfcs_stack).any(), np.isnan(scores_stack[pair_id//5]).any())
    print(np.isfinite(isfcs_stack).all(), np.isfinite(scores_stack[pair_id//5]).all())'''

    clf.fit(isfc_train, scores_stack[train]) #[:,pair_id//5]) #train
    #print('fit')
    
    score = clf.score(isfc_test, scores_stack[test]) #[:,pair_id//5])
    #scores.append(score)
    '''pred = clf.predict(results_test)
    r = pearsonr(pred, scores_stack[test, pair_id//5])[0]
    scores.append(r)'''
    print(score)
    print(clf.alpha_)
    best_alphas_scores[f] = clf.alpha_
    coefs_scores[f] = clf.coef_
    results_scores[f] = score
    print(f"Finished regression {f}") 

<class 'sklearn.model_selection._split.KFold'>
(2048,)


1it [00:58, 58.88s/it]

0.8455196930310321
1000
Finished regression 0


1it [01:06, 66.20s/it]


KeyboardInterrupt: 

In [178]:
results_scores

array([0.84551969, 0.84470921, 0.84577218, 0.85310874, 0.87957426,
       0.83576738, 0.83353471, 0.85206643, 0.80810978, 0.76510637,
       0.80496414, 0.89672606, 0.8512236 , 0.79649948, 0.79999911,
       0.82034636, 0.86156215, 0.83832692, 0.8680621 , 0.82136696,
       0.8403249 , 0.87252633, 0.81681889, 0.82835036, 0.81140365,
       0.87233491, 0.71630214, 0.84139658, 0.78481169, 0.85837856,
       0.74764467, 0.80548001])

In [172]:
np.argmax(coefs_scores[0])
coefs_scores[0,737]

0.04058669253059696

In [136]:
alphas = [100, 1000, 10000] #0.1, 1, 10 removed for speed on reruns - initial tests had 1000 consistently best

outerCV = KFold(n_splits=32) 
innerCV = KFold(n_splits=31) 

scaler = StandardScaler()

print(type(outerCV))

teams = (0,5)
isfcs_stack = np.vstack((isfcs[:, teams[0], :], isfcs[:, teams[1], :]))#.squeeze()[:, np.newaxis]
scores_stack = np.concatenate((score_stack[:, 0], score_stack[:, 1]), axis=0)
print(scores_stack.shape)

for f, (train, test) in tqdm(enumerate(outerCV.split(isfcs_stack))):
    if f<17:
        continue
    
    clf = RidgeCV(alphas, cv=innerCV) 
    
    isfc_train = scaler.fit_transform(isfcs_stack[train]) 
    isfc_test = scaler.transform(isfcs_stack[test])
    
    #print('fit next')

    '''print(isfcs_stack.shape)
    print(np.isnan(isfcs_stack).any(), np.isnan(scores_stack[pair_id//5]).any())
    print(np.isfinite(isfcs_stack).all(), np.isfinite(scores_stack[pair_id//5]).all())'''

    clf.fit(isfc_train, scores_stack[train]) #[:,pair_id//5]) #train
    #print('fit')
    
    score = clf.score(isfc_test, scores_stack[test]) #[:,pair_id//5])
    #scores.append(score)
    '''pred = clf.predict(results_test)
    r = pearsonr(pred, scores_stack[test, pair_id//5])[0]
    scores.append(r)'''
    print(score)
    print(clf.alpha_)
    best_alphas_scores[f] = clf.alpha_
    coefs_scores[f] = clf.coef_
    results_scores[f] = score
    print(f"Finished regression {f}") 

<class 'sklearn.model_selection._split.KFold'>
(2048,)


18it [01:04,  3.57s/it]

0.8383269176699449
1000
Finished regression 17


19it [02:10,  8.15s/it]

0.8680620988764362
1000
Finished regression 18


...

0.8583785553932528
1000
Finished regression 29


31it [15:26, 63.26s/it]

0.7476446685282234
1000
Finished regression 30


32it [16:33, 31.04s/it]

0.805480012115756
1000
Finished regression 31


In [144]:
np.save('isfcs_scores_results.npy', results_scores)
np.save('isfcs_scores_coefs.npy', coefs_scores)
np.save('isfcs_scores_alphas.npy', best_alphas_scores)

In [127]:
results_scores = np.load('isfcs_scores_results.npy')
coefs_scores = np.load('isfcs_scores_coefs.npy')
best_alphas_scores = np.load('isfcs_scores_alphas.npy')

In [173]:
isfcs_stack.shape

(2048, 10153)

In [176]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

#use set of alphas = [0, 1, 10, 100, 1000, 10000]
Cs = [0.1, 1, 10, 100, 1000, 10000]

outerCV = KFold(n_splits=32) 
innerCV = KFold(n_splits=31) 
#cv = PredefinedSplit(map_ids)

scaler = StandardScaler()

print(type(outerCV))

teams = (0,5)
isfcs_stack = np.vstack((isfcs[:, teams[0], :], isfcs[:, teams[1], :]))#.squeeze()[:, np.newaxis]
print(isfcs_stack.shape)
wins_stack = np.concatenate((win_stack[:, 0], win_stack[:, 1]), axis=0)
print(wins_stack.shape)
#raise

results_wins = np.full((n_maps), np.nan)   #n_repeats
coefs_wins = np.full((n_maps, pc_pairs), np.nan)
best_Cs_wins = np.full((n_maps), np.nan)
pairs = [(0, (0,1)), (5, (2,3))]
for f, (train, test) in tqdm(enumerate(outerCV.split(isfcs_stack))):
    clf = LogisticRegressionCV(Cs=Cs, cv=innerCV) 
    
    isfc_train = scaler.fit_transform(isfcs_stack[train]) 
    isfc_test = scaler.transform(isfcs_stack[test])

    #print('next fit')
    clf.fit(isfc_train, wins_stack[train]) 
    #print('fit')
    
    score = clf.score(isfc_test, wins_stack[test]) 
    print(score)
    print(clf.C_)
    best_Cs_wins[f] = clf.C_
    coefs_wins[f] = clf.coef_
    results_wins[f] = score
    print(f"Finished regression {f}") 

<class 'sklearn.model_selection._split.KFold'>
(1024,)
(2048,)


0it [00:00, ?it/s]

1984


RuntimeError: No active exception to reraise

In [182]:
np.mean(results_wins)

0.93115234375

In [181]:
best_Cs_wins

array([1.e+02, 1.e+02, 1.e+02, 1.e+00, 1.e-01, 1.e-01, 1.e-01, 1.e+00,
       1.e+02, 1.e-01, 1.e+00, 1.e+03, 1.e+01, 1.e-01, 1.e-01, 1.e-01,
       1.e+00, 1.e-01, 1.e+01, 1.e+03, 1.e+03, 1.e+00, 1.e+03, 1.e+03,
       1.e-01, 1.e+03, 1.e+00, 1.e+03, 1.e-01, 1.e-01, 1.e+03, 1.e+02])

In [158]:
np.save('isfcs_wins_results.npy', results_wins)
np.save('isfcs_wins_coefs.npy', coefs_wins)
np.save('isfcs_wins_Cs.npy', best_Cs_wins)

In [180]:
results_wins = np.load('isfcs_wins_results.npy')
coefs_wins = np.load('isfcs_wins_coefs.npy')
best_Cs_wins = np.load('isfcs_wins_Cs.npy')

In [151]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import PredefinedSplit
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score
from scipy.stats import zscore
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

#use set of alphas = [0, 1, 10, 100, 1000, 10000]
alphas = [100, 1000, 10000] #0.1, 1, 10 removed for speed on reruns - initial tests had 1000 consistently best

outerCV = KFold(n_splits=32) 
innerCV = KFold(n_splits=31) 
#cv = PredefinedSplit(map_ids)

scaler = StandardScaler()

print(type(outerCV))

teams = (0,5)
isfc_diffs = isfcs[:, teams[0], :] - isfcs[:, teams[1], :]#.squeeze()[:, np.newaxis]
#print(win_stack[:, 0].shape)
score_diffs = score_stack[:, 0] - score_stack[:, 1]
print(scores_stack.shape)
#raise

results_score_diffs = np.full((n_maps), np.nan)   #n_repeats
coefs_score_diffs = np.full((n_maps, pc_pairs), np.nan)
best_alphas_score_diffs = np.full((n_maps), np.nan)
pairs = [(0, (0,1)), (5, (2,3))]
#for pc_pair in np.arange(pc_pairs):
#for pair_id, pair in pairs:
#scores = []
for f, (train, test) in tqdm(enumerate(outerCV.split(isfc_diffs))):
    clf = RidgeCV(alphas, cv=innerCV) 
    
    isfc_train = scaler.fit_transform(isfc_diffs[train]) 
    isfc_test = scaler.transform(isfc_diffs[test])
    
    #print('fit next')

    '''print(isfcs_stack.shape)
    print(np.isnan(isfcs_stack).any(), np.isnan(scores_stack[pair_id//5]).any())
    print(np.isfinite(isfcs_stack).all(), np.isfinite(scores_stack[pair_id//5]).all())'''

    clf.fit(isfc_train, score_diffs[train]) #[:,pair_id//5]) #train
    #print('fit')
    
    score = clf.score(isfc_test, score_diffs[test]) #[:,pair_id//5])
    #scores.append(score)
    '''pred = clf.predict(results_test)
    r = pearsonr(pred, scores_stack[test, pair_id//5])[0]
    scores.append(r)'''
    print(score)
    print(clf.alpha_)
    best_alphas_score_diffs[f] = clf.alpha_
    coefs_score_diffs[f] = clf.coef_
    results_score_diffs[f] = score
    print(f"Finished regression {f}") 

<class 'sklearn.model_selection._split.KFold'>
(2048,)


1it [00:16, 16.41s/it]

0.7913235863720571
1000
Finished regression 0


2it [00:32, 16.49s/it]

0.6827646695183975
1000
Finished regression 1


...

0.5080424658622837
1000
Finished regression 29


31it [08:35, 16.28s/it]

0.69715710526683
1000
Finished regression 30


32it [08:52, 16.65s/it]

0.6752569850353056
1000
Finished regression 31


In [167]:
results_score_diffs

array([0.79132359, 0.68276467, 0.62474952, 0.76767795, 0.81413451,
       0.74346133, 0.83555682, 0.74379396, 0.89410498, 0.71628897,
       0.65622158, 0.60285949, 0.69702823, 0.66936491, 0.5561819 ,
       0.73730845, 0.76437419, 0.7487459 , 0.6428299 , 0.64171205,
       0.63650393, 0.75320305, 0.7761259 , 0.81228686, 0.72073542,
       0.5889016 , 0.79112127, 0.72040866, 0.77236135, 0.50804247,
       0.69715711, 0.67525699])

In [156]:
np.save('isfcs_score_diffs_results.npy', results_score_diffs)
np.save('isfcs_score_diffs_coefs.npy', coefs_score_diffs)
np.save('isfcs_score_diffs_alphas.npy', best_alphas_score_diffs)

In [162]:
results_score_diffs = np.load('isfcs_score_diffs_results.npy')
coefs_score_diffs = np.load('isfcs_score_diffs_coefs.npy')
best_alphas_score_diffs = np.load('isfcs_score_diffs_alphas.npy')

In [ ]:
#############################OLD#############################################

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import PredefinedSplit
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score
from scipy.stats import zscore
from sklearn.model_selection import KFold

#use set of alphas = [0, 1, 10, 100, 1000, 10000]
alphas = [0.1, 1, 10, 100, 1000, 10000]

outerCV = KFold(n_splits=32) 
innerCV = KFold(n_splits=31) 
#cv = PredefinedSplit(map_ids)

results = np.full((pc_pairs, 2), np.nan)   #n_repeats
coefs = np.full((pc_pairs, 2), np.nan)
alphas = np.full((pc_pairs, 2), np.nan)
pairs = [(0, (0,1)), (5, (2,3))]
for pc_pair in np.arange(pc_pairs):
    for pair_id, pair in pairs:
        scores = []
        for f, (train, test) in enumerate(cv.split()):
            clf = RidgeCV(alphas, cv=innerCV) 

            isfcs_stack = isfcs[:, pair_id, pc_pair].squeeze()[:, np.newaxis]
            #results_train = zscore(isfcs_stack) 
            #results_test = zscore(isfcs_stack)

            '''print(isfcs_stack.shape)
            print(np.isnan(isfcs_stack).any(), np.isnan(scores_stack[pair_id//5]).any())
            print(np.isfinite(isfcs_stack).all(), np.isfinite(scores_stack[pair_id//5]).all())'''

            clf.fit(isfcs_stack, scores_stack[:,pair_id//5]) #train
            score = clf.score(isfcs_stack, scores_stack[:,pair_id//5])
            '''pred = clf.predict(results_test)
            r = pearsonr(pred, scores_stack[test, pair_id//5])[0]
            scores.append(r)'''
            #print(scores)
            print(clf.alpha_)
            alphas[pc_pair, pair_id//5] = clf.alpha_
            coefs[pc_pair, pair_id//5] = clf.coef_
            results[pc_pair, pair_id//5] = score
    if pc_pair%100==0:
        print(f"Finished regression {pc_pair}") 